In [6]:
pip install matplotlib

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------- ----------- 5.8/8.1 MB 28.6 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 26.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 15.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 17.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

# BEA API Key
API_KEY = "83E1BA00-3ADE-4A5A-BF65-45A8E6AACC99"

# BEA API URL
BEA_URL = "https://apps.bea.gov/api/data"

# Parameters for Real GDP (RGDP) in Chained Dollars (Table 1.1.6)
rgdp_params = {
    "UserID": API_KEY,
    "method": "GetData",
    "datasetname": "NIPA",
    "TableName": "T10106",  # Table 1.1.6 for Real GDP (Chained Dollars)
    "Frequency": "A",  # Annual Data
    "Year": "ALL",  # Fetch all available years
    "ResultFormat": "json"
}

# Parameters for GDP Deflator (Table 1.1.9, Line 1: Gross Domestic Product Implicit Price Deflator)
gdp_deflator_params = {
    "UserID": API_KEY,
    "method": "GetData",
    "datasetname": "NIPA",
    "TableName": "T10109",  # Table 1.1.9 for GDP Implicit Price Deflator
    "Frequency": "A",  # Annual Data
    "Year": "ALL",  # Fetch all available years
    "ResultFormat": "json"
}

# Fetch Real GDP Data
rgdp_response = requests.get(BEA_URL, params=rgdp_params)
rgdp_data = rgdp_response.json()

# Fetch GDP Deflator Data
gdp_deflator_response = requests.get(BEA_URL, params=gdp_deflator_params)
gdp_deflator_data = gdp_deflator_response.json()

# Extract RGDP Data
rgdp_records = []
for item in rgdp_data["BEAAPI"]["Results"]["Data"]:
    if item["LineNumber"] == "1":  # Line 1 represents GDP in chained dollars
        rgdp_records.append({
            "Year": int(item["TimePeriod"]),  # Use TimePeriod for Year
            "RGDP": float(item["DataValue"].replace(",", ""))  # Remove commas
        })

df_rgdp = pd.DataFrame(rgdp_records)

# Extract GDP Deflator Data
gdp_deflator_records = []
for item in gdp_deflator_data["BEAAPI"]["Results"]["Data"]:
    if item["LineNumber"] == "1":  # Line 1 represents GDP Deflator
        gdp_deflator_records.append({
            "Year": int(item["TimePeriod"]),  # Use TimePeriod for Year
            "GDP Deflator": float(item["DataValue"].replace(",", ""))  # Remove commas
        })

df_gdp_deflator = pd.DataFrame(gdp_deflator_records)

# Merge the datasets on Year
df_adas = pd.merge(df_rgdp, df_gdp_deflator, on="Year")

# Save to CSV for Tableau
df_adas.to_csv("adas_gdp_deflator.csv", index=False)

df_adas.head()

,Year,RGDP,GDP Deflator
0,1929,1191124.0,8.778
1,1930,1089785.0,8.457
2,1931,1019977.0,7.587
3,1932,888414.0,6.700
4,1933,877431.0,6.514
